In [14]:
# Load environment variables from project root
from pathlib import Path
from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / '.env'
    if _env.is_file():
        load_dotenv(_env)
        break

In [15]:
# Create Databricks Spark session
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
# spark.sql('CREATE SCHEMA IF NOT EXISTS ddc_databricks.gold')

In [16]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, DoubleType

# Silver feature tables (one row per package)
silver_github = spark.table('ddc_databricks.silver.github_package_features')
silver_pypi = spark.table('ddc_databricks.silver.pypi_package_features')
silver_so = spark.table('ddc_databricks.silver.so_package_features')

# Silver text tables for sentiment analysis
silver_so_questions = spark.table('ddc_databricks.silver.so_questions')
silver_so_answers = spark.table('ddc_databricks.silver.so_answers')
silver_readmes = spark.table('ddc_databricks.silver.github_readmes')
silver_pypi_metadata = spark.table('ddc_databricks.silver.pypi_metadata')

In [17]:
# Sentiment analysis UDF — tries VADER, then TextBlob, then keyword heuristic.

sentiment_schema = StructType([
    StructField('compound', DoubleType(), True),
    StructField('positive', DoubleType(), True),
    StructField('negative', DoubleType(), True),
    StructField('neutral', DoubleType(), True),
])

_SENTIMENT_MAX_CHARS = 5000

def _compute_sentiment(text):
    if text is None or len(str(text).strip()) == 0:
        return (0.0, 0.0, 0.0, 1.0)
    snippet = str(text)[:_SENTIMENT_MAX_CHARS]

    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        scores = SentimentIntensityAnalyzer().polarity_scores(snippet)
        return (float(scores['compound']), float(scores['pos']),
                float(scores['neg']), float(scores['neu']))
    except ImportError:
        print("Vader sentiment not available")
        pass

    try:
        from textblob import TextBlob
        p = TextBlob(snippet).sentiment.polarity
        return (float(p), float(max(0.0, p)),
                float(max(0.0, -p)), float(1.0 - abs(p)))
    except ImportError:
        print("TextBlob sentiment not available")
        pass

    lower = snippet.lower()
    _POS = ['good', 'great', 'excellent', 'easy', 'fast', 'useful', 'best',
            'perfect', 'works', 'love', 'helpful', 'nice', 'clean', 'awesome']
    _NEG = ['bad', 'error', 'bug', 'slow', 'broken', 'fail', 'crash',
            'wrong', 'issue', 'problem', 'terrible', 'awful', 'hate', 'worst']
    pos = sum(1 for w in _POS if w in lower)
    neg = sum(1 for w in _NEG if w in lower)
    total = pos + neg
    if total == 0:
        return (0.0, 0.0, 0.0, 1.0)
    compound = (pos - neg) / total
    return (float(compound), float(pos / total),
            float(neg / total), float(1.0 - abs(compound)))

sentiment_udf = F.udf(_compute_sentiment, sentiment_schema)

In [18]:
# 1) Sentiment analysis on all text sources and save in gold.package_sentiment

so_q_sent = (
    silver_so_questions
    .select('package_name', 'question_id', 'question_body_text')
    .withColumn('s', sentiment_udf(F.col('question_body_text')))
    .groupBy('package_name')
    .agg(
        F.avg('s.compound').alias('so_question_sentiment_avg'),
        F.avg('s.positive').alias('so_question_sentiment_pos_avg'),
        F.avg('s.negative').alias('so_question_sentiment_neg_avg'),
        F.count('question_id').alias('so_question_sentiment_count'),
    )
)

so_a_sent = (
    silver_so_answers
    .select('package_name', 'answer_id', 'answer_body_text')
    .withColumn('s', sentiment_udf(F.col('answer_body_text')))
    .groupBy('package_name')
    .agg(
        F.avg('s.compound').alias('so_answer_sentiment_avg'),
        F.avg('s.positive').alias('so_answer_sentiment_pos_avg'),
        F.avg('s.negative').alias('so_answer_sentiment_neg_avg'),
        F.count('answer_id').alias('so_answer_sentiment_count'),
    )
)

readme_sent = (
    silver_readmes
    .select('package_name', 'readme_clean_text')
    .withColumn('s', sentiment_udf(F.col('readme_clean_text')))
    .select('package_name',
            F.col('s.compound').alias('readme_sentiment_compound'),
            F.col('s.positive').alias('readme_sentiment_pos'),
            F.col('s.negative').alias('readme_sentiment_neg'))
)

pypi_sent = (
    silver_pypi_metadata
    .select('package_name', 'description_clean_text')
    .withColumn('s', sentiment_udf(F.col('description_clean_text')))
    .select('package_name',
            F.col('s.compound').alias('pypi_desc_sentiment_compound'),
            F.col('s.positive').alias('pypi_desc_sentiment_pos'),
            F.col('s.negative').alias('pypi_desc_sentiment_neg'))
)

# Weighted overall: SO questions 30%, SO answers 30%, README 25%, PyPI desc 15%
package_sentiment = (
    so_q_sent.alias('q')
    .join(so_a_sent.alias('a'), on='package_name', how='full')
    .join(readme_sent.alias('r'), on='package_name', how='full')
    .join(pypi_sent.alias('d'), on='package_name', how='full')
    .withColumn(
        'overall_sentiment',
        F.coalesce(F.col('so_question_sentiment_avg'), F.lit(0.0)) * 0.30
        + F.coalesce(F.col('so_answer_sentiment_avg'), F.lit(0.0)) * 0.30
        + F.coalesce(F.col('readme_sentiment_compound'), F.lit(0.0)) * 0.25
        + F.coalesce(F.col('pypi_desc_sentiment_compound'), F.lit(0.0)) * 0.15
    )
    .withColumn('scored_at', F.current_timestamp())
)

(
    package_sentiment
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable('ddc_databricks.gold.package_sentiment')
)

package_sentiment.select(
    'package_name', 'so_question_sentiment_avg', 'so_answer_sentiment_avg',
    'readme_sentiment_compound', 'pypi_desc_sentiment_compound', 'overall_sentiment'
).orderBy('package_name').show(truncate=False)

+------------+-------------------------+-----------------------+-------------------------+----------------------------+---------------------+
|package_name|so_question_sentiment_avg|so_answer_sentiment_avg|readme_sentiment_compound|pypi_desc_sentiment_compound|overall_sentiment    |
+------------+-------------------------+-----------------------+-------------------------+----------------------------+---------------------+
|django      |-0.11955555555555555     |-0.02743290043290043   |0.0                      |0.0                         |-0.044096536796536794|
|fastapi     |0.2411111111111112       |0.21446428571428572    |0.14285714285714285      |0.14285714285714285         |0.1938154761904762   |
|flask       |-0.2604497354497355      |-0.12308333333333334   |0.3333333333333333       |0.3333333333333333          |0.018273412698412685 |
|numpy       |-0.09755555555555555     |0.06799107142857143    |-0.5                     |-0.5                        |-0.20886934523809525 |
|panda